### Simple Gen AI app to summarize a webpage

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACTING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

**Reading the content with bs4**

Data Ingestion

In [2]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/concepts/")

In [4]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://python.langchain.com/v0.2/docs/concepts/', 'title': 'Conceptual guide | 🦜️🔗 LangChain', 'description': 'This section contains introductions to key parts of LangChain.', 'language': 'en'}, page_content='\n\n\n\n\nConceptual guide | 🦜️🔗 LangChain\n\n\n\n\n\n\n\nSkip to main contentA newer LangChain version is out! Check out the latest version.IntegrationsAPI referenceLatestLegacyMorePeopleContributingCookbooks3rd party tutorialsYouTubearXivv0.2Latestv0.2v0.1🦜️🔗LangSmithLangSmith DocsLangChain HubJS/TS Docs💬SearchIntroductionTutorialsBuild a Question Answering application over a Graph DatabaseTutorialsBuild a Simple LLM Application with LCELBuild a Query Analysis SystemBuild a ChatbotConversational RAGBuild an Extraction ChainBuild an AgentTaggingdata_generationBuild a Local RAG ApplicationBuild a PDF ingestion and Question/Answering systemBuild a Retrieval Augmented Generation (RAG) AppVector stores and retrieversBuild a Question/Answering system ov

Splitting the document into chunks

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
)
docs_split = text_splitter.split_documents(docs)
docs_split

[Document(metadata={'source': 'https://python.langchain.com/v0.2/docs/concepts/', 'title': 'Conceptual guide | 🦜️🔗 LangChain', 'description': 'This section contains introductions to key parts of LangChain.', 'language': 'en'}, page_content='Conceptual guide | 🦜️🔗 LangChain'),
 Document(metadata={'source': 'https://python.langchain.com/v0.2/docs/concepts/', 'title': 'Conceptual guide | 🦜️🔗 LangChain', 'description': 'This section contains introductions to key parts of LangChain.', 'language': 'en'}, page_content='Skip to main contentA newer LangChain version is out! Check out the latest version.IntegrationsAPI referenceLatestLegacyMorePeopleContributingCookbooks3rd party tutorialsYouTubearXivv0.2Latestv0.2v0.1🦜️🔗LangSmithLangSmith DocsLangChain HubJS/TS Docs💬SearchIntroductionTutorialsBuild a Question Answering application over a Graph DatabaseTutorialsBuild a Simple LLM Application with LCELBuild a Query Analysis SystemBuild a ChatbotConversational RAGBuild an Extraction ChainBuild an 

**Converting split documents into vectors**

In [8]:
from langchain_community.embeddings import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="gemma:2b")

C:\Users\arnav\AppData\Local\Temp\ipykernel_30996\899155049.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="gemma:2b")


In [19]:
embeddings.embed_documents(docs_split)

KeyboardInterrupt: 

In [6]:
from langchain_community.vectorstores import FAISS


In [21]:
db= FAISS.from_documents(docs_split,embeddings)

In [22]:
response = db.similarity_search_with_score("Langcchain key components")
print(response)

[(Document(metadata={'source': 'https://python.langchain.com/v0.2/docs/concepts/', 'title': 'Conceptual guide | 🦜️🔗 LangChain', 'description': 'This section contains introductions to key parts of LangChain.', 'language': 'en'}, page_content='reliable implementations use features like tool calling to reliably format outputs\nand reduce variance.Please see the LangGraph documentation for more information,'), 2615.871), (Document(metadata={'source': 'https://python.langchain.com/v0.2/docs/concepts/', 'title': 'Conceptual guide | 🦜️🔗 LangChain', 'description': 'This section contains introductions to key parts of LangChain.', 'language': 'en'}, page_content='Generally, such models are better at tool calling than non-fine-tuned models, and are recommended for use cases that require tool calling.'), 2771.054), (Document(metadata={'source': 'https://python.langchain.com/v0.2/docs/concepts/', 'title': 'Conceptual guide | 🦜️🔗 LangChain', 'description': 'This section contains introductions to key

In [23]:
db.save_local("langchain_concepts_db")

: 

In [10]:
saved_db=FAISS.load_local("langchain_concepts_db",embeddings=embeddings,allow_dangerous_deserialization=True)

In [12]:
query = "Langschain key components"
result = saved_db.similarity_search(query=query)
result[0].page_content

'reliable implementations use features like tool calling to reliably format outputs\nand reduce variance.Please see the LangGraph documentation for more information,'

### Retrieval Chains
Think of retrievers as an interface for getting the relevant contextual data from a vectorstore DB.
We can convert our vectorstoreDB into a retriever


In [13]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    """Answer the questions based on given context:<context>{context}</context>Question:{input}"""
)

In [21]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4")

In [22]:
from langchain.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='Answer the questions based on given context:<context>{context}</context>Question:{input}'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x0000020AA64A8770>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x0000020AA64A8830>, root_client=<openai.OpenAI object at 0x0000020AA50E0050>, root_async_client=<openai.AsyncOpenAI object at 0x0000020AA64A84A0>, model_name='gpt-4', model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_docu

In [23]:
from langchain_core.documents import Document
document_chain.invoke(
    {
        "input":"Langchain provides the following",
        "context":[Document(page_content = "Langchain provides the following key components that make it extremely flexible to integrate GPT models to create AI powered applications")]
    }
)

'key components that make it extremely flexible to integrate GPT models to create AI powered applications.'

In [24]:
retriever_from_db = saved_db.as_retriever()
retriever_from_db

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020A427815B0>, search_kwargs={})

In [26]:
from langchain.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever_from_db, document_chain)

In [29]:
response = retrieval_chain.invoke(
    {
        "input":"What packages does langchain as a framework provide?",
       # "context":[Document(page_content = "Langchain provides the following key components that make it extremely flexible to integrate GPT models to create AI powered applications")]
    }
)

In [30]:
print(response["answer"])

LangChain as a framework provides the following packages: langchain-core, which contains base abstractions of different components and ways to compose them together, and langchain-community, which contains third party integrations that are maintained by the LangChain community. The main langchain package also contains chains, agents, and retrieval strategies that make up an application's cognitive architecture.
